In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("HeartFailureBigDataPipeline") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()


In [2]:
df = spark.read.csv("heart_failure_clinical_records_dataset.csv",
                    header=True,
                    inferSchema=True)

print("Row Count:", df.count())
print("Column Count:", len(df.columns))


Row Count: 299
Column Count: 13


In [3]:
from pyspark.sql.functions import col, count, when

df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()


+---+-------+------------------------+--------+-----------------+-------------------+---------+----------------+------------+---+-------+----+-----------+
|age|anaemia|creatinine_phosphokinase|diabetes|ejection_fraction|high_blood_pressure|platelets|serum_creatinine|serum_sodium|sex|smoking|time|DEATH_EVENT|
+---+-------+------------------------+--------+-----------------+-------------------+---------+----------------+------------+---+-------+----+-----------+
|  0|      0|                       0|       0|                0|                  0|        0|               0|           0|  0|      0|   0|          0|
+---+-------+------------------------+--------+-----------------+-------------------+---------+----------------+------------+---+-------+----+-----------+



In [4]:
df.cache()


DataFrame[age: double, anaemia: int, creatinine_phosphokinase: int, diabetes: int, ejection_fraction: int, high_blood_pressure: int, platelets: double, serum_creatinine: double, serum_sodium: int, sex: int, smoking: int, time: int, DEATH_EVENT: int]

In [5]:
from pyspark.ml.feature import VectorAssembler, StandardScaler

feature_cols = df.columns[:-1]

assembler = VectorAssembler(inputCols=feature_cols,
                            outputCol="raw_features")

assembled_df = assembler.transform(df)

scaler = StandardScaler(inputCol="raw_features",
                        outputCol="features")

scaled_df = scaler.fit(assembled_df).transform(assembled_df)

data = scaled_df.select("features", "DEATH_EVENT")


In [6]:
train_df, test_df = data.randomSplit([0.8, 0.2], seed=42)
train_df.persist()


DataFrame[features: vector, DEATH_EVENT: int]

In [7]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(labelCol="DEATH_EVENT")

lr_model = lr.fit(train_df)
lr_pred = lr_model.transform(test_df)


In [8]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(labelCol="DEATH_EVENT", numTrees=100)

rf_model = rf.fit(train_df)
rf_pred = rf_model.transform(test_df)


In [9]:
from pyspark.ml.classification import DecisionTreeClassifier

dt = DecisionTreeClassifier(labelCol="DEATH_EVENT")

dt_model = dt.fit(train_df)
dt_pred = dt_model.transform(test_df)


In [10]:
from pyspark.ml.classification import GBTClassifier

gbt = GBTClassifier(labelCol="DEATH_EVENT")

gbt_model = gbt.fit(train_df)
gbt_pred = gbt_model.transform(test_df)
